In [1]:
DATASET = "nyc_tlc_yellow"
SILVER_PATH = f"../lakehouse/silver/{DATASET}"
GOLD_PATH = f"../lakehouse/gold/{DATASET}"

print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)

SILVER_PATH: ../lakehouse/silver/nyc_tlc_yellow
GOLD_PATH: ../lakehouse/gold/nyc_tlc_yellow


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYC-TLC-Lakehouse")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .getOrCreate()
)

spark

26/02/23 20:59:13 WARN Utils: Your hostname, willian-A320M-S2H resolves to a loopback address: 127.0.1.1; using 192.168.0.110 instead (on interface enp6s0)
26/02/23 20:59:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/willian/PhaifferTech/nyc-tlc-lakehouse/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/willian/.ivy2/cache
The jars for the packages stored in: /home/willian/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3069fb09-512a-4580-b9cf-623c1b103940;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 169ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   

In [3]:
df_silver = spark.read.format("delta").load(SILVER_PATH)
print("silver_rows:", df_silver.count())
df_silver.select("pickup_date").distinct().orderBy("pickup_date").show(5, truncate=False)

26/02/23 20:59:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


silver_rows: 2839435


26/02/23 20:59:30 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+-----------+
|pickup_date|
+-----------+
|2024-01-01 |
|2024-01-02 |
|2024-01-03 |
|2024-01-04 |
|2024-01-05 |
+-----------+
only showing top 5 rows



In [4]:
from pyspark.sql.functions import (
    col, unix_timestamp, when, coalesce, lit
)

df_enriched = (
    df_silver
    .withColumn(
        "trip_duration_sec",
        unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))
    )
    .withColumn("trip_duration_min", col("trip_duration_sec") / 60.0)
    # revenue components (treat nulls as zero)
    .withColumn("fare_amount_z", coalesce(col("fare_amount"), lit(0.0)))
    .withColumn("tip_amount_z", coalesce(col("tip_amount"), lit(0.0)))
    .withColumn("tolls_amount_z", coalesce(col("tolls_amount"), lit(0.0)))
    .withColumn("extra_z", coalesce(col("extra"), lit(0.0)))
    .withColumn("mta_tax_z", coalesce(col("mta_tax"), lit(0.0)))
    .withColumn("improvement_surcharge_z", coalesce(col("improvement_surcharge"), lit(0.0)))
    .withColumn("congestion_surcharge_z", coalesce(col("congestion_surcharge"), lit(0.0)))
    .withColumn("airport_fee_z", coalesce(col("Airport_fee"), lit(0.0)))
    .withColumn(
        "total_revenue",
        col("fare_amount_z")
        + col("tip_amount_z")
        + col("tolls_amount_z")
        + col("extra_z")
        + col("mta_tax_z")
        + col("improvement_surcharge_z")
        + col("congestion_surcharge_z")
        + col("airport_fee_z")
    )
)

df_enriched.select("pickup_date", "trip_duration_min", "total_revenue").show(5, truncate=False)

+-----------+------------------+------------------+
|pickup_date|trip_duration_min |total_revenue     |
+-----------+------------------+------------------+
|2024-01-13 |2.1166666666666667|15.52             |
|2024-01-13 |10.533333333333333|15.91             |
|2024-01-13 |13.75             |23.18             |
|2024-01-13 |12.533333333333333|21.470000000000002|
|2024-01-13 |20.366666666666667|34.67             |
+-----------+------------------+------------------+
only showing top 5 rows



In [5]:
from pyspark.sql.functions import avg, sum, count, round as sround

kpi_daily = (
    df_enriched
    .filter(col("trip_duration_sec") > 0)
    .groupBy("pickup_date")
    .agg(
        count(lit(1)).alias("trips"),
        sum("fare_amount_z").alias("total_fare"),
        sum("tip_amount_z").alias("total_tip"),
        sum("total_revenue").alias("total_revenue"),
        avg("trip_distance").alias("avg_trip_distance"),
        avg("trip_duration_min").alias("avg_trip_duration_min"),
        avg("passenger_count").alias("avg_passenger_count"),
    )
    .withColumn("avg_trip_distance", sround(col("avg_trip_distance"), 3))
    .withColumn("avg_trip_duration_min", sround(col("avg_trip_duration_min"), 3))
    .withColumn("avg_passenger_count", sround(col("avg_passenger_count"), 3))
)

kpi_daily.orderBy("pickup_date").show(10, truncate=False)

+-----------+-----+------------------+------------------+------------------+-----------------+---------------------+-------------------+
|pickup_date|trips|total_fare        |total_tip         |total_revenue     |avg_trip_distance|avg_trip_duration_min|avg_passenger_count|
+-----------+-----+------------------+------------------+------------------+-----------------+---------------------+-------------------+
|2024-01-01 |74830|1662841.4800000603|256689.6099999985 |2338765.5199999292|4.665            |16.689               |1.568              |
|2024-01-02 |72341|1548093.8100000108|259731.07999999783|2272526.359999874 |4.218            |17.063               |1.432              |
|2024-01-03 |79188|1590700.8200000047|271635.90999999957|2357063.259999927 |3.956            |16.677               |1.395              |
|2024-01-04 |99066|1860543.840000016 |332253.1699999981 |2801594.289999928 |3.367            |16.175               |1.371              |
|2024-01-05 |99019|1799324.4899999816|325

In [6]:
KPI_DAILY_PATH = f"{GOLD_PATH}/kpi_daily_overview"

(
    kpi_daily
    .write
    .format("delta")
    .mode("overwrite")
    .save(KPI_DAILY_PATH)
)

print("Saved:", KPI_DAILY_PATH)

Saved: ../lakehouse/gold/nyc_tlc_yellow/kpi_daily_overview


In [7]:
df_gold = spark.read.format("delta").load(KPI_DAILY_PATH)
print("gold_days:", df_gold.count())
df_gold.orderBy("pickup_date").show(31, truncate=False)

gold_days: 31
+-----------+------+------------------+------------------+------------------+-----------------+---------------------+-------------------+
|pickup_date|trips |total_fare        |total_tip         |total_revenue     |avg_trip_distance|avg_trip_duration_min|avg_passenger_count|
+-----------+------+------------------+------------------+------------------+-----------------+---------------------+-------------------+
|2024-01-01 |74830 |1662841.4800000603|256689.6099999985 |2338765.5199999292|4.665            |16.689               |1.568              |
|2024-01-02 |72341 |1548093.8100000108|259731.07999999783|2272526.359999874 |4.218            |17.063               |1.432              |
|2024-01-03 |79188 |1590700.8200000047|271635.90999999957|2357063.259999927 |3.956            |16.677               |1.395              |
|2024-01-04 |99066 |1860543.840000016 |332253.1699999981 |2801594.289999928 |3.367            |16.175               |1.371              |
|2024-01-05 |99019 |